# 02. Model selection on development data

Candidates are scored by random split, spatial CV and dataset holdout. The highest mean of spatial CV and dataset holdout is selected. Test parcels are not used.

**Input:** current inputs. **Output:** `current_selection.json`. **Next:** 03.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import pandas as pd
from IPython.display import display
from model_aza.improved import workflow
pd.set_option("display.width", 200)

## Candidates

In [ ]:
pd.DataFrame([{"candidate": name} for name in workflow.CANDIDATES])

## Score and select

In [ ]:
inputs_handoff = ""  # Optional inputs receipt; blank uses the current pointer
receipt = workflow.select(root, inputs_handoff or None, progress=lambda m: print(m, flush=True))
selection = json.loads((receipt.parent / "selection.json").read_text())
scores = pd.read_csv(receipt.parent / "candidate_scores.csv")
display(scores.groupby("candidate")[["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1"]].mean().round(3))
display(pd.DataFrame(selection["summary"]).T.round(3))
print("Selected:", selection["selected"], "| rule:", selection["rule"])